In [1]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Shuffle Hash Join y Broadcast Hash Join").getOrCreate()

# Estrategias de Optimización de *Joins* en Apache Spark

La operación de *Join* es una de las más costosas y que más recursos consume en Apache Spark debido a la necesidad de mover datos a través de la red. A alto nivel, existen dos estrategias principales que el motor de Spark utiliza para unir dos conjuntos de datos: **Shuffle Hash Join** y **Broadcast Hash Join**.

El criterio principal para seleccionar una estrategia en particular se basa en el **tamaño de los dos conjuntos de datos**.

---

## 1. Shuffle Hash Join (Para conjuntos de datos grandes)

Esta estrategia se selecciona automáticamente cuando **ambos conjuntos de datos son grandes** y ninguno de ellos cabe en la memoria de un solo ejecutor.

Conceptualmente, para combinar las filas de dos conjuntos de datos que cumplen con la condición de unión, Spark necesita que las filas que comparten la misma clave se ubiquen físicamente en la misma partición. Su implementación consta de dos pasos:

### Pasos del Shuffle Hash Join:
1. **Fase de Shuffle (Mezcla y Redistribución):** Spark calcula el valor *hash* de las columnas de la expresión de *join* para cada fila en ambos conjuntos de datos. Luego, mueve esas filas a través de la red hacia una partición común. Para determinar exactamente a qué partición enviar una fila, Spark realiza una operación aritmética simple: `valor_hash % número_de_particiones`.
2. **Fase de Hash Join:** Una vez que los datos con las mismas claves están en la misma partición, Spark construye una tabla hash en memoria con las filas de uno de los lados y recorre el otro lado para combinar las columnas correspondientes.



> ⚠️ **Advertencia de Rendimiento:** Al mover datos a través de la red (fase de *Shuffle*), estos pasan por un proceso intensivo de **serialización y deserialización**, lo que consume mucha CPU y ancho de banda. Por esta razón, se debe reducir la frecuencia con la que se unen grandes conjuntos de datos siempre que sea posible.

---

## 2. Broadcast Hash Join (Para cuando un conjunto de datos es pequeño)

Esta estrategia (también conocida como *Map-Side Join*) es aplicable únicamente cuando **uno de los conjuntos de datos es lo suficientemente pequeño** como para caber por completo en la memoria RAM de los ejecutores (el límite por defecto en Spark es de 10 MB, configurable mediante `spark.sql.autoBroadcastJoinThreshold`).

Sabiendo que el *Shuffle* es una operación costosa, la estrategia de *Broadcast* evita por completo el movimiento y la redistribución del conjunto de datos grande. En su lugar, solo mueve el conjunto de datos pequeño mediante los siguientes dos pasos:

### Pasos del Broadcast Hash Join:
1. **Fase de Transmisión (*Broadcast*):** Spark envía una copia completa del conjunto de datos más pequeño a cada una de las particiones donde reside el conjunto de datos más grande (es decir, a todos los ejecutores del clúster).
2. **Fase de Unión Local:** Cada ejecutor realiza un *join* local recorriendo las filas de su sección del conjunto de datos grande y buscando de manera inmediata las coincidencias en la copia local del conjunto pequeño.



---

## Conclusión

Es evidente que **Broadcast Hash Join es la estrategia preferida siempre que sea posible**, ya que elimina por completo el costo del *Shuffle* en la red para el dataset de gran tamaño, transformando una operación distribuida compleja en un conjunto de búsquedas locales ultrarrápidas.




In [6]:
departamentos = spark.read.parquet("./data/departamentos")

In [7]:
empleados = spark.read.parquet("./data/empleados")

In [4]:
from pyspark.sql.functions import col, broadcast

In [8]:
# tiene que cumplir con ser un df pequeño para que pueda ser enviado a los ejecutores

empleados.join(broadcast(departamentos), col("num_dpto") == col("id")).show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 33|    derecho|
| Katia|      33| 33|    derecho|
|  Raul|      34| 34| matemática|
| Laura|      34| 34| matemática|
|Sandro|      31| 31|     letras|
+------+--------+---+-----------+



In [9]:
# como sabemos que realmente hizo un bradcast?  con la funcion explain

empleados.join(broadcast(departamentos), col("num_dpto") == col("id")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [num_dpto#7L], [id#4L], Inner, BuildRight, false
   :- Filter isnotnull(num_dpto#7L)
   :  +- FileScan parquet [nombre#6,num_dpto#7L] Batched: true, DataFilters: [isnotnull(num_dpto#7L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/data/empleados], PartitionFilters: [], PushedFilters: [IsNotNull(num_dpto)], ReadSchema: struct<nombre:string,num_dpto:bigint>
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=108]
      +- Filter isnotnull(id#4L)
         +- FileScan parquet [id#4L,nombre_dpto#5] Batched: true, DataFilters: [isnotnull(id#4L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/data/departamentos], PartitionFilters: [], PushedFilters: [IsNotNull(id)], ReadSchema: struct<id:bigint,nombre_dpto:string>


